# Cross-platform and Strong Feature Vector Linear Regression Models

This notebook trains and compares linear regression models using cross-platform platform features and a stronger feature vector set to predict stream counts.

In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

DATA_PATH = Path('../data/spotify-2023-cleaned.csv')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(df.head())
print(f"\nLoaded shape: {df.shape}")

                            track_name    artist(s)_name  artist_count  \
0  Seven (feat. Latto) (Explicit Ver.)  Latto, Jung Kook             2   
1                                 LALA       Myke Towers             1   
2                              vampire    Olivia Rodrigo             1   
3                         Cruel Summer      Taylor Swift             1   
4                       WHERE SHE GOES         Bad Bunny             1   

   released_year  released_month  released_day  in_spotify_playlists  \
0           2023               7            14                   553   
1           2023               3            23                  1474   
2           2023               6            30                  1397   
3           2019               8            23                  7858   
4           2023               5            18                  3133   

   in_spotify_charts      streams  in_apple_playlists  ...  bpm  key   mode  \
0                147  141381703.0          

In [2]:
# Prepare a helper for training

def train_and_evaluate(feature_vector, target='streams'):
    X = df[feature_vector].copy()
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    return model, {
        'features': feature_vector,
        'mse': mse,
        'rmse': rmse,
        'r2': r2,
    }

In [3]:
# Cross-platform feature vector
cross_platform_features = [
    'in_apple_playlists',
    'in_apple_charts',
    'in_deezer_charts',
    'in_deezer_playlists',
    'in_shazam_charts'
]
cross_platform_features = [col for col in cross_platform_features if col in df.columns]

print("Cross-platform features:")
print(cross_platform_features)

cross_model, cross_metrics = train_and_evaluate(cross_platform_features)
print("\nCross-platform model metrics:")
print(cross_metrics)

joblib.dump(cross_model, MODEL_DIR / 'spotify_cross_platform_lrmodel.pkl')
print(f"Saved cross-platform model to {MODEL_DIR / 'spotify_cross_platform_lrmodel.pkl'}")

Cross-platform features:
['in_apple_playlists', 'in_apple_charts', 'in_deezer_charts', 'in_deezer_playlists', 'in_shazam_charts']

Cross-platform model metrics:
{'features': ['in_apple_playlists', 'in_apple_charts', 'in_deezer_charts', 'in_deezer_playlists', 'in_shazam_charts'], 'mse': 1.0587899890159579e+17, 'rmse': np.float64(325390532.90099853), 'r2': 0.6091988484347389}
Saved cross-platform model to ..\models\spotify_cross_platform_lrmodel.pkl


In [4]:
# Stronger feature vector based on high-correlation platform indicators
strong_features = [
    'in_spotify_playlists',
    'in_spotify_charts',
    'in_apple_playlists',
    'in_apple_charts',
    'in_deezer_charts',
    'in_deezer_playlists',
    'in_shazam_charts'
]
strong_features = [col for col in strong_features if col in df.columns]

print("Strong feature vector:")
print(strong_features)

strong_model, strong_metrics = train_and_evaluate(strong_features)
print("\nStrong feature vector model metrics:")
print(strong_metrics)

joblib.dump(strong_model, MODEL_DIR / 'spotify_strong_fvs_lrmodel.pkl')
print(f"Saved strong-feature model to {MODEL_DIR / 'spotify_strong_fvs_lrmodel.pkl'}")

Strong feature vector:
['in_spotify_playlists', 'in_spotify_charts', 'in_apple_playlists', 'in_apple_charts', 'in_deezer_charts', 'in_deezer_playlists', 'in_shazam_charts']

Strong feature vector model metrics:
{'features': ['in_spotify_playlists', 'in_spotify_charts', 'in_apple_playlists', 'in_apple_charts', 'in_deezer_charts', 'in_deezer_playlists', 'in_shazam_charts'], 'mse': 8.159954126929989e+16, 'rmse': np.float64(285656334.2012564), 'r2': 0.6988147316648003}
Saved strong-feature model to ..\models\spotify_strong_fvs_lrmodel.pkl


In [1]:
# Compare both models side-by-side
comparison = pd.DataFrame([
    {'model': 'cross_platform', **cross_metrics},
    {'model': 'strong_fvs', **strong_metrics}
])

print(comparison[['model', 'rmse', 'r2']])

NameError: name 'pd' is not defined